## top 100 pyspark questions asked to data engineer for experience 2-3 years

In [1]:
#Basic Questions for practice
spark.stop()

In [2]:
#1.How to initialize spark session
from pyspark.sql import SparkSession

spark=SparkSession.builder \
      .appName("Practice App") \
      .enableHiveSupport() \
      .master("local[*]") \
      .getOrCreate()

#this creates new or gets existing spark session if present.
#master{local(*)} uses all nodes available.

25/01/19 09:37:51 INFO SparkEnv: Registering MapOutputTracker
25/01/19 09:37:51 INFO SparkEnv: Registering BlockManagerMaster
25/01/19 09:37:51 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
25/01/19 09:37:51 INFO SparkEnv: Registering OutputCommitCoordinator


## Easy 40 Questions PySpark

In [13]:
#2.Create a PySpark DataFrame from a Python dictionary.
from pyspark.sql.types import *
data = [
    {"name": "Alice", "age": 25},
    {"name": "Bob", "age": 30},
    {"name": "Charlie", "age": 35}
]
df=spark.createDataFrame(data)
df.show()
row_count = df.count()
print(f"Number of rows: {row_count}")

+---+-------+
|age|   name|
+---+-------+
| 25|  Alice|
| 30|    Bob|
| 35|Charlie|
+---+-------+

Number of rows: 3


In [14]:
#3. Display Schema of DataFrame
df.printSchema()

root
 |-- age: long (nullable = true)
 |-- name: string (nullable = true)



In [15]:
#4. Load a CSV file into dataframe
path="/user/duggiedogy/data/e-commerce-data.csv"
df1=spark.read.csv(path,inferSchema=True,header=True)
df1.show(5)

+--------+-----------+----------+-----------------+----------+--------+-----+
|order_id|customer_id|order_date|order_purchase_ts|product_id|quantity|price|
+--------+-----------+----------+-----------------+----------+--------+-----+
|    1001|        501|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|
|    1002|        502|02-10-2017| 02-10-2017 10:56|      P002|       1| 85.5|
|    1003|        503|03-10-2017| 03-10-2017 11:45|      P001|       3|150.0|
|    1004|        504|04-10-2017| 04-10-2017 14:15|      P003|       1|200.0|
|    1005|        505|05-10-2017| 05-10-2017 16:40|      P002|       5| 85.5|
+--------+-----------+----------+-----------------+----------+--------+-----+
only showing top 5 rows



In [7]:
#5. Select specific columns from a DataFrame
# df2=df1.select("order_id")
# df2.show(5)

In [6]:
#6. Filter orders where the price is greater than 100 and placed after 2017-10-10
from pyspark.sql.functions import *

filtered_df = df1.filter((col("price") > 100) & (col("order_date") > "2017-10-10"))
filtered_df.show()

#Calculate total revenue per product.
revenue_df=df1.groupBy('product_id').agg(sum(col('price')*col('quantity')).alias('total_revenue'))
revenue_df.show()

+--------+-----------+----------+-----------------+----------+--------+-----+
|order_id|customer_id|order_date|order_purchase_ts|product_id|quantity|price|
+--------+-----------+----------+-----------------+----------+--------+-----+
+--------+-----------+----------+-----------------+----------+--------+-----+



+----------+-------------+
|product_id|total_revenue|
+----------+-------------+
|      P003|        600.0|
|      P004|        550.0|
|      P005|        300.0|
|      P006|        360.0|
|      P002|       1368.0|
|      P001|       1500.0|
+----------+-------------+



In [7]:
#Add a column discounted_price (10% off if price > 100).
discount_df=df1.withColumn('discounted_price',when(col("price") > 100, col("price") * 0.9).otherwise(col("price")))
discount_df.show(5)

+--------+-----------+----------+-----------------+----------+--------+-----+----------------+
|order_id|customer_id|order_date|order_purchase_ts|product_id|quantity|price|discounted_price|
+--------+-----------+----------+-----------------+----------+--------+-----+----------------+
|    1001|        501|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|           135.0|
|    1002|        502|02-10-2017| 02-10-2017 10:56|      P002|       1| 85.5|            85.5|
|    1003|        503|03-10-2017| 03-10-2017 11:45|      P001|       3|150.0|           135.0|
|    1004|        504|04-10-2017| 04-10-2017 14:15|      P003|       1|200.0|           180.0|
|    1005|        505|05-10-2017| 05-10-2017 16:40|      P002|       5| 85.5|            85.5|
+--------+-----------+----------+-----------------+----------+--------+-----+----------------+
only showing top 5 rows



In [8]:
#Get the top 3 most ordered products by quantity.
top_3_df=df1.groupBy("product_id").agg(sum("quantity").alias("total_quantity")) \
                    .orderBy(desc("total_quantity")) \
                    .limit(3)
top_3_df.show()

#Find the average order value per customer.
avg_order_value_df = df1.groupBy("customer_id").agg(
    avg(col("price") * col("quantity")).alias("avg_order_value")
)
avg_order_value_df.show(5)

#Filter orders placed between '2017-10-12' and '2017-10-16'.
filtered_date_df = df1.filter((col("order_date") >= "2017-10-12") & (col("order_date") <= "2017-10-16"))
filtered_date_df.show()

#Identify the day with the highest total sales.
sales_per_day_df = df1.groupBy("order_date").agg(
    sum(col("price") * col("quantity")).alias("daily_sales")
).orderBy(desc("daily_sales"))

sales_per_day_df.show(1)

+----------+--------------+
|product_id|total_quantity|
+----------+--------------+
|      P002|            16|
|      P001|            10|
|      P004|             5|
+----------+--------------+

+-----------+---------------+
|customer_id|avg_order_value|
+-----------+---------------+
|        505|          427.5|
|        509|          400.0|
|        513|          120.0|
|        510|          342.0|
|        501|          300.0|
+-----------+---------------+
only showing top 5 rows

+--------+-----------+----------+-----------------+----------+--------+-----+
|order_id|customer_id|order_date|order_purchase_ts|product_id|quantity|price|
+--------+-----------+----------+-----------------+----------+--------+-----+
+--------+-----------+----------+-----------------+----------+--------+-----+

+----------+-----------+
|order_date|daily_sales|
+----------+-----------+
|16-10-2017|      600.0|
+----------+-----------+
only showing top 1 row



In [9]:
#Calculate the cumulative quantity of P001 over time.
from pyspark.sql.window import Window

window_spec=Window.orderBy('order_date').rowsBetween(Window.unboundedPreceding, Window.currentRow)

cummulative_df=df1.filter(col("product_id") == "P001").withColumn("cumulative_quantity", sum("quantity").over(window_spec))
cummulative_df.show()

25/01/08 16:11:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 16:11:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 16:11:54 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 16:11:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 16:11:55 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+--------+-----------+----------+-----------------+----------+--------+-----+-------------------+
|order_id|customer_id|order_date|order_purchase_ts|product_id|quantity|price|cumulative_quantity|
+--------+-----------+----------+-----------------+----------+--------+-----+-------------------+
|    1001|        501|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|                  2|
|    1003|        503|03-10-2017| 03-10-2017 11:45|      P001|       3|150.0|                  5|
|    1007|        507|09-10-2017| 09-10-2017 08:20|      P001|       1|150.0|                  6|
|    1014|        514|16-10-2017| 16-10-2017 19:00|      P001|       4|150.0|                 10|
+--------+-----------+----------+-----------------+----------+--------+-----+-------------------+



In [6]:
#map vs flatMap (on RDD for demonstration)
rdd = spark.sparkContext.parallelize(["apple orange", "banana", "grape mango"])
map_result = rdd.map(lambda x: x.split(" "))
flatmap_result = rdd.flatMap(lambda x: x.split(" "))
#show results using actions
print("Map Result:", map_result.collect())
print("FlatMap Result:", flatmap_result.collect())

Map Result: [['apple', 'orange'], ['banana'], ['grape', 'mango']]
FlatMap Result: ['apple', 'orange', 'banana', 'grape', 'mango']


In [10]:
from pyspark.sql.functions import *
# 16. Remove duplicate rows
df1.dropDuplicates().show()

# 17. collect vs show
print(df1.collect())  # Collect returns list of Row objects
df1.show(5)  # Show displays top 5 rows

# 18. Write DataFrame to CSV
df1.write.csv("/user/duggiedogy/data/orders_output.csv", header=True, mode="overwrite")

# 19. Filter null values
filtered_df = df1.filter(col("price").isNotNull())
filtered_df.show()

# 20. na.drop() vs na.fill()
df1.na.drop().show()  # Drop rows with any nulls
df1.na.fill({"price": 0, "quantity": 1}).show()  # Fill nulls in price and quantity

+--------+-----------+----------+-----------------+----------+--------+-----+
|order_id|customer_id|order_date|order_purchase_ts|product_id|quantity|price|
+--------+-----------+----------+-----------------+----------+--------+-----+
|    1011|        511|13-10-2017| 13-10-2017 13:50|      P006|       1|120.0|
|    1016|        516|18-10-2017| 18-10-2017 10:55|      P002|       6| 85.5|
|    1009|        509|11-10-2017| 11-10-2017 09:40|      P003|       2|200.0|
|    1012|        512|14-10-2017| 14-10-2017 15:10|      P004|       3|110.0|
|    1004|        504|04-10-2017| 04-10-2017 14:15|      P003|       1|200.0|
|    1007|        507|09-10-2017| 09-10-2017 08:20|      P001|       1|150.0|
|    1003|        503|03-10-2017| 03-10-2017 11:45|      P001|       3|150.0|
|    1001|        501|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|
|    1005|        505|05-10-2017| 05-10-2017 16:40|      P002|       5| 85.5|
|    1014|        514|16-10-2017| 16-10-2017 19:00|      P001|  

In [10]:
# 21. Rename a column
renamed_df = df1.withColumnRenamed("order_price", "price")
renamed_df.show()

+--------+-----------+----------+-----------------+----------+--------+-----+
|order_id|customer_id|order_date|order_purchase_ts|product_id|quantity|price|
+--------+-----------+----------+-----------------+----------+--------+-----+
|    1001|        501|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|
|    1002|        502|02-10-2017| 02-10-2017 10:56|      P002|       1| 85.5|
|    1003|        503|03-10-2017| 03-10-2017 11:45|      P001|       3|150.0|
|    1004|        504|04-10-2017| 04-10-2017 14:15|      P003|       1|200.0|
|    1005|        505|05-10-2017| 05-10-2017 16:40|      P002|       5| 85.5|
|    1006|        506|08-10-2017| 08-10-2017 12:30|      P004|       2|110.0|
|    1007|        507|09-10-2017| 09-10-2017 08:20|      P001|       1|150.0|
|    1008|        508|10-10-2017| 10-10-2017 18:05|      P005|       3| 60.0|
|    1009|        509|11-10-2017| 11-10-2017 09:40|      P003|       2|200.0|
|    1010|        510|12-10-2017| 12-10-2017 11:30|      P002|  

In [3]:
#21. join two DataFrames
customers_df=spark.read.csv('/user/duggiedogy/data/customers.csv',header=True,inferSchema=True)
joined_df=df1.join(customers_df, on="customer_id", how="inner")
joined_df.show()

+-----------+--------+----------+-----------------+----------+--------+-----+-------------+--------+
|customer_id|order_id|order_date|order_purchase_ts|product_id|quantity|price|customer_name|location|
+-----------+--------+----------+-----------------+----------+--------+-----+-------------+--------+
+-----------+--------+----------+-----------------+----------+--------+-----+-------------+--------+



In [15]:
# 23. INNER JOIN vs LEFT JOIN
inner_join_df = df1.join(customers_df, on="customer_id", how="inner")
inner_join_df.show()
left_join_df = df1.join(customers_df, on="customer_id", how="left")
left_join_df.show()

+-----------+--------+----------+-----------------+----------+--------+-----+-------------+--------+
|customer_id|order_id|order_date|order_purchase_ts|product_id|quantity|price|customer_name|location|
+-----------+--------+----------+-----------------+----------+--------+-----+-------------+--------+
+-----------+--------+----------+-----------------+----------+--------+-----+-------------+--------+

+-----------+--------+----------+-----------------+----------+--------+-----+-------------+--------+
|customer_id|order_id|order_date|order_purchase_ts|product_id|quantity|price|customer_name|location|
+-----------+--------+----------+-----------------+----------+--------+-----+-------------+--------+
|        501|    1001|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|         NULL|    NULL|
|        502|    1002|02-10-2017| 02-10-2017 10:56|      P002|       1| 85.5|         NULL|    NULL|
|        503|    1003|03-10-2017| 03-10-2017 11:45|      P001|       3|150.0|         NULL

In [22]:
from pyspark.sql.functions import *
# 27. Replace values in a column
#replaced_df = df1.withColumn("status", col("status").replace("pending", "completed"))
#replaced_df.show()

# 28. Cast column to different data type
casted_df = df1.withColumn("price", col("price").cast("double"))
casted_df.show()

# 29. Calculate max value in a column
max_price_df = df1.agg(max("price").alias("max_price"))
max_price_df.show()

# 30. withColumnRenamed example
renamed_df = df1.withColumnRenamed("quantity", "order_quantity")
print("renamed dataframe")
renamed_df.show()

+--------+-----------+----------+-----------------+----------+--------+-----+
|order_id|customer_id|order_date|order_purchase_ts|product_id|quantity|price|
+--------+-----------+----------+-----------------+----------+--------+-----+
|    1001|        501|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|
|    1002|        502|02-10-2017| 02-10-2017 10:56|      P002|       1| 85.5|
|    1003|        503|03-10-2017| 03-10-2017 11:45|      P001|       3|150.0|
|    1004|        504|04-10-2017| 04-10-2017 14:15|      P003|       1|200.0|
|    1005|        505|05-10-2017| 05-10-2017 16:40|      P002|       5| 85.5|
|    1006|        506|08-10-2017| 08-10-2017 12:30|      P004|       2|110.0|
|    1007|        507|09-10-2017| 09-10-2017 08:20|      P001|       1|150.0|
|    1008|        508|10-10-2017| 10-10-2017 18:05|      P005|       3| 60.0|
|    1009|        509|11-10-2017| 11-10-2017 09:40|      P003|       2|200.0|
|    1010|        510|12-10-2017| 12-10-2017 11:30|      P002|  

In [11]:
# 31. What is an RDD and how is it different from DataFrames?
# RDD (Resilient Distributed Dataset) is a low-level API that represents distributed collections of objects.
# DataFrames provide a higher-level abstraction with schema and SQL-like operations.

In [16]:
# 32. Create RDD from a Python list
rdd_list=spark.sparkContext.parallelize([1,2,3,4,5])
rdd_list.collect()

[1, 2, 3, 4, 5]

In [22]:
# 33. Basic map and reduce operations
mapped_list=rdd_list.map(lambda x: x*2)
print("map output: ", mapped_list.collect())
reduce_list=mapped_list.reduce(lambda a,b:a+b)
print("reduce output: ",reduce_list)

map output:  [2, 4, 6, 8, 10]
reduce output:  30


In [24]:
# 34. Persist an RDD in memory
persisted_rdd = rdd_list.persist()

# 35. cache vs persist
# cache() stores the RDD in memory, persist() can store at different levels (memory, disk, or both)

In [26]:
# 36. Read text files into RDDs
text_rdd = spark.sparkContext.textFile("/user/duggiedogy/data/sample_text.txt")
text_rdd.collect()

['apple banana orange',
 'grape mango apple',
 'banana orange mango',
 'apple grape']

In [30]:
# 37. Perform word count using RDD
#flatMap splits each line into words and flattens the result, producing one RDD with all the words.
#map transforms each word into a tuple (word, 1), setting the stage for aggregation.
#reduceByKey aggregates the counts by summing up values for the same key (word).
word_count=text_rdd.flatMap(lambda x: x.split(" ")).map(lambda x:(x,1)).reduceByKey(lambda a,b:a+b)
word_count.collect()

[('apple', 3), ('banana', 2), ('orange', 2), ('grape', 2), ('mango', 2)]

In [34]:
# 38. Repartition an RDD
# Check the number of partitions
print("Number of partitions before repartitioning:", rdd_list.getNumPartitions())
repartitioned_rdd = rdd_list.repartition(4)
print("Number of partitions after repartitioning:", repartitioned_rdd.getNumPartitions())

# 39. coalesce vs repartition
# coalesce reduces partitions without shuffling, repartition reshuffles data, which can be more expensive.

Number of partitions before repartitioning: 2
Number of partitions after repartitioning: 4


In [37]:
# 40. Convert RDD to DataFrame
#map(lambda x: (x,)) to format each element as a tuple (x,).
rdd_to_df=rdd_list.map(lambda x:(x,)).toDF(['number'])
rdd_to_df.show()

+------+
|number|
+------+
|     1|
|     2|
|     3|
|     4|
|     5|
+------+



## Medium Complexity PySpark 35 Questions

In [39]:
# 41. Calculate running total using window functions
window_spec=Window.orderBy('order_date')
running_total_df=df1.withColumn('running_sum',sum("price").over(window_spec))
running_total_df.show(2)

25/01/08 16:44:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 16:44:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 16:44:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 16:44:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 16:44:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+--------+-----------+----------+-----------------+----------+--------+-----+-----------+
|order_id|customer_id|order_date|order_purchase_ts|product_id|quantity|price|running_sum|
+--------+-----------+----------+-----------------+----------+--------+-----+-----------+
|    1001|        501|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|      150.0|
|    1002|        502|02-10-2017| 02-10-2017 10:56|      P002|       1| 85.5|      235.5|
+--------+-----------+----------+-----------------+----------+--------+-----+-----------+
only showing top 2 rows



In [44]:
# 42. Rank rows within partitions
# 43. Rank vs dense_rank vs row_number
rank_df=df1.withColumn("ranking",rank().over(window_spec))
rank_df.show(2)

25/01/08 16:48:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 16:48:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 16:48:19 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+--------+-----------+----------+-----------------+----------+--------+-----+-------+
|order_id|customer_id|order_date|order_purchase_ts|product_id|quantity|price|ranking|
+--------+-----------+----------+-----------------+----------+--------+-----+-------+
|    1001|        501|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|      1|
|    1002|        502|02-10-2017| 02-10-2017 10:56|      P002|       1| 85.5|      2|
+--------+-----------+----------+-----------------+----------+--------+-----+-------+
only showing top 2 rows



In [47]:
dense_rank=df1.withColumn("dense_rank",dense_rank().over(window_spec))
dense_rank.show(2)

row_number=df1.withColumn("row_number",row_number().over(window_spec))
row_number.show(2)

TypeError: 'DataFrame' object is not callable

In [49]:
# 44. Calculate moving average
moving_avg=df1.withColumn('moving_avg',avg('price').over(window_spec))
moving_avg.show(5)

25/01/08 16:50:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 16:50:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 16:50:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+--------+-----------+----------+-----------------+----------+--------+-----+----------+
|order_id|customer_id|order_date|order_purchase_ts|product_id|quantity|price|moving_avg|
+--------+-----------+----------+-----------------+----------+--------+-----+----------+
|    1001|        501|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|     150.0|
|    1002|        502|02-10-2017| 02-10-2017 10:56|      P002|       1| 85.5|    117.75|
|    1003|        503|03-10-2017| 03-10-2017 11:45|      P001|       3|150.0|     128.5|
|    1004|        504|04-10-2017| 04-10-2017 14:15|      P003|       1|200.0|   146.375|
|    1005|        505|05-10-2017| 05-10-2017 16:40|      P002|       5| 85.5|     134.2|
+--------+-----------+----------+-----------------+----------+--------+-----+----------+
only showing top 5 rows



25/01/08 16:50:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 16:50:23 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [53]:
# 45. Perform cumulative sum
cumm_avg=df1.withColumn('cummulative_sum',sum('price').over(window_spec))
cumm_avg.show(5)

25/01/08 16:53:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 16:53:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 16:53:58 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 16:53:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 16:53:59 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+--------+-----------+----------+-----------------+----------+--------+-----+---------------+
|order_id|customer_id|order_date|order_purchase_ts|product_id|quantity|price|cummulative_sum|
+--------+-----------+----------+-----------------+----------+--------+-----+---------------+
|    1001|        501|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|          150.0|
|    1002|        502|02-10-2017| 02-10-2017 10:56|      P002|       1| 85.5|          235.5|
|    1003|        503|03-10-2017| 03-10-2017 11:45|      P001|       3|150.0|          385.5|
|    1004|        504|04-10-2017| 04-10-2017 14:15|      P003|       1|200.0|          585.5|
|    1005|        505|05-10-2017| 05-10-2017 16:40|      P002|       5| 85.5|          671.0|
+--------+-----------+----------+-----------------+----------+--------+-----+---------------+
only showing top 5 rows



In [52]:
# 46. Partition data using window functions
partition_window=Window.partitionBy('customer_id').orderBy('order_date')
partition_df=df1.withColumn('rank',rank().over(partition_window))
partition_df.show()

+--------+-----------+----------+-----------------+----------+--------+-----+----+
|order_id|customer_id|order_date|order_purchase_ts|product_id|quantity|price|rank|
+--------+-----------+----------+-----------------+----------+--------+-----+----+
|    1001|        501|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|   1|
|    1002|        502|02-10-2017| 02-10-2017 10:56|      P002|       1| 85.5|   1|
|    1003|        503|03-10-2017| 03-10-2017 11:45|      P001|       3|150.0|   1|
|    1004|        504|04-10-2017| 04-10-2017 14:15|      P003|       1|200.0|   1|
|    1005|        505|05-10-2017| 05-10-2017 16:40|      P002|       5| 85.5|   1|
|    1006|        506|08-10-2017| 08-10-2017 12:30|      P004|       2|110.0|   1|
|    1007|        507|09-10-2017| 09-10-2017 08:20|      P001|       1|150.0|   1|
|    1008|        508|10-10-2017| 10-10-2017 18:05|      P005|       3| 60.0|   1|
|    1009|        509|11-10-2017| 11-10-2017 09:40|      P003|       2|200.0|   1|
|   

In [55]:
# 47. Find first and last rows within each partition
first_last_df = df1.withColumn("first_order", first("price").over(partition_window))
first_last_df.show()

last_df = df1.withColumn("last_order", last("price").over(partition_window))
last_df.show()

+--------+-----------+----------+-----------------+----------+--------+-----+-----------+
|order_id|customer_id|order_date|order_purchase_ts|product_id|quantity|price|first_order|
+--------+-----------+----------+-----------------+----------+--------+-----+-----------+
|    1001|        501|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|      150.0|
|    1002|        502|02-10-2017| 02-10-2017 10:56|      P002|       1| 85.5|       85.5|
|    1003|        503|03-10-2017| 03-10-2017 11:45|      P001|       3|150.0|      150.0|
|    1004|        504|04-10-2017| 04-10-2017 14:15|      P003|       1|200.0|      200.0|
|    1005|        505|05-10-2017| 05-10-2017 16:40|      P002|       5| 85.5|       85.5|
|    1006|        506|08-10-2017| 08-10-2017 12:30|      P004|       2|110.0|      110.0|
|    1007|        507|09-10-2017| 09-10-2017 08:20|      P001|       1|150.0|      150.0|
|    1008|        508|10-10-2017| 10-10-2017 18:05|      P005|       3| 60.0|       60.0|
|    1009|

In [ ]:
emp_df=spark.read.csv('/user/duggiedogy/data/employee_dataset.csv',header=True,inferSchema=True)
emp_df.show()

In [58]:
# 48. Find the Nth highest salary per department
from pyspark.sql.functions import *
window_nth=Window.partitionBy('department_id').orderBy(col('salary').desc())
nth_df=emp_df.withColumn('nth highest',row_number().over(window_nth))
nth_df.show()

+-----------+-------------+------+-----------+
|employee_id|department_id|salary|nth highest|
+-----------+-------------+------+-----------+
|        110|            1|  7800|          1|
|        103|            1|  7500|          2|
|        102|            1|  6000|          3|
|        101|            1|  5000|          4|
|        104|            2|  8000|          1|
|        105|            2|  7200|          2|
|        111|            2|  7100|          3|
|        106|            2|  6600|          4|
|        112|            3|  9400|          1|
|        109|            3|  9100|          2|
|        107|            3|  9000|          3|
|        108|            3|  8800|          4|
+-----------+-------------+------+-----------+



In [60]:
# 49. Apply lag and lead functions
#lag(column, offset) – Fetches the value from a previous row at the specified offset. Default offset is 1.
#lead(column, offset) – Fetches the value from the next row at the specified offset.
lag_df=df1.withColumn('lag function',lag('price').over(window_spec))
lag_df.show()

lead_df=lag_df.withColumn('lead function',lead('price').over(window_spec))
lead_df.show()

25/01/08 17:09:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 17:09:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 17:09:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 17:09:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 17:09:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+--------+-----------+----------+-----------------+----------+--------+-----+------------+
|order_id|customer_id|order_date|order_purchase_ts|product_id|quantity|price|lag function|
+--------+-----------+----------+-----------------+----------+--------+-----+------------+
|    1001|        501|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|        NULL|
|    1002|        502|02-10-2017| 02-10-2017 10:56|      P002|       1| 85.5|       150.0|
|    1003|        503|03-10-2017| 03-10-2017 11:45|      P001|       3|150.0|        85.5|
|    1004|        504|04-10-2017| 04-10-2017 14:15|      P003|       1|200.0|       150.0|
|    1005|        505|05-10-2017| 05-10-2017 16:40|      P002|       5| 85.5|       200.0|
|    1006|        506|08-10-2017| 08-10-2017 12:30|      P004|       2|110.0|        85.5|
|    1007|        507|09-10-2017| 09-10-2017 08:20|      P001|       1|150.0|       110.0|
|    1008|        508|10-10-2017| 10-10-2017 18:05|      P005|       3| 60.0|       150.0|

25/01/08 17:09:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 17:09:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 17:09:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 17:09:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 17:09:30 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+--------+-----------+----------+-----------------+----------+--------+-----+------------+-------------+
|order_id|customer_id|order_date|order_purchase_ts|product_id|quantity|price|lag function|lead function|
+--------+-----------+----------+-----------------+----------+--------+-----+------------+-------------+
|    1001|        501|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|        NULL|         85.5|
|    1002|        502|02-10-2017| 02-10-2017 10:56|      P002|       1| 85.5|       150.0|        150.0|
|    1003|        503|03-10-2017| 03-10-2017 11:45|      P001|       3|150.0|        85.5|        200.0|
|    1004|        504|04-10-2017| 04-10-2017 14:15|      P003|       1|200.0|       150.0|         85.5|
|    1005|        505|05-10-2017| 05-10-2017 16:40|      P002|       5| 85.5|       200.0|        110.0|
|    1006|        506|08-10-2017| 08-10-2017 12:30|      P004|       2|110.0|        85.5|        150.0|
|    1007|        507|09-10-2017| 09-10-2017 08:20|    

In [61]:
# 49. Apply lag and lead functions
orders_updated=spark.read.csv('/user/duggiedogy/data/orders_updated.csv',header=True,inferSchema=True)
orders_updated.show()

+--------+----------+-----+
|order_id|order_date|price|
+--------+----------+-----+
|       1|01-01-2024|  500|
|       2|02-01-2024|  600|
|       3|03-01-2024|  450|
|       4|04-01-2024|  700|
|       5|05-01-2024|  650|
+--------+----------+-----+



In [64]:
# Define window to partition by and order by date
window_order=Window.orderBy('order_date')
# Apply lag to calculate previous day's sales
sales_diff=orders_updated.withColumn('prev_day_sales',lag('price').over(window_order))
sales_diff.show()
# Calculate the difference
sales_diff=sales_diff.withColumn('sales_diff',col('price')-col('prev_day_sales'))
print("The lag function brings the previous day's sales into the current row.")
sales_diff.show()

25/01/08 17:21:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 17:21:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 17:21:36 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 17:21:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 17:21:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+--------+----------+-----+--------------+
|order_id|order_date|price|prev_day_sales|
+--------+----------+-----+--------------+
|       1|01-01-2024|  500|          NULL|
|       2|02-01-2024|  600|           500|
|       3|03-01-2024|  450|           600|
|       4|04-01-2024|  700|           450|
|       5|05-01-2024|  650|           700|
+--------+----------+-----+--------------+

The lag function brings the previous day's sales into the current row.


25/01/08 17:21:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 17:21:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 17:21:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


+--------+----------+-----+--------------+----------+
|order_id|order_date|price|prev_day_sales|sales_diff|
+--------+----------+-----+--------------+----------+
|       1|01-01-2024|  500|          NULL|      NULL|
|       2|02-01-2024|  600|           500|       100|
|       3|03-01-2024|  450|           600|      -150|
|       4|04-01-2024|  700|           450|       250|
|       5|05-01-2024|  650|           700|       -50|
+--------+----------+-----+--------------+----------+



25/01/08 17:21:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/01/08 17:21:37 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.


In [67]:
# 50. Handle duplicate records using window functions
duplicates_window = Window.partitionBy("customer_id", "order_date").orderBy(col("order_id"))
deduped_df = df1.withColumn("row_number", row_number().over(duplicates_window)).filter(col("row_number") == 1)
deduped_df.show()

+--------+-----------+----------+-----------------+----------+--------+-----+----------+
|order_id|customer_id|order_date|order_purchase_ts|product_id|quantity|price|row_number|
+--------+-----------+----------+-----------------+----------+--------+-----+----------+
|    1001|        501|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|         1|
|    1002|        502|02-10-2017| 02-10-2017 10:56|      P002|       1| 85.5|         1|
|    1003|        503|03-10-2017| 03-10-2017 11:45|      P001|       3|150.0|         1|
|    1004|        504|04-10-2017| 04-10-2017 14:15|      P003|       1|200.0|         1|
|    1005|        505|05-10-2017| 05-10-2017 16:40|      P002|       5| 85.5|         1|
|    1006|        506|08-10-2017| 08-10-2017 12:30|      P004|       2|110.0|         1|
|    1007|        507|09-10-2017| 09-10-2017 08:20|      P001|       1|150.0|         1|
|    1008|        508|10-10-2017| 10-10-2017 18:05|      P005|       3| 60.0|         1|
|    1009|        509

In [4]:
#Explain how to perform a broadcast join in PySpark.
'''
A broadcast join is used when one of the DataFrames is small enough to fit in memory. 
Spark broadcasts the smaller DataFrame to all the executor nodes, avoiding shuffle operations and improving performance.
'''

from pyspark.sql.functions import *
broadcast_join_df = df1.join(broadcast(customers_df), on="customer_id", how="inner")
broadcast_join_df.show()

+-----------+--------+----------+-----------------+----------+--------+-----+-------------+--------+
|customer_id|order_id|order_date|order_purchase_ts|product_id|quantity|price|customer_name|location|
+-----------+--------+----------+-----------------+----------+--------+-----+-------------+--------+
+-----------+--------+----------+-----------------+----------+--------+-----+-------------+--------+



In [5]:
#52. How do you optimize join performance in PySpark?
'''
Optimization Techniques:

Broadcast smaller DataFrame when one DataFrame is significantly smaller.
Filter data early to reduce the size of the DataFrames before joining.
Partitioning: Ensure the data is properly partitioned on the join key.
Avoid Skewness by salting or balancing the data.
Cache intermediate results if they are reused.
Use SQL hints like /*+ BROADCAST() */ to control the join strategy.
'''

'\nOptimization Techniques:\n\nBroadcast smaller DataFrame when one DataFrame is significantly smaller.\nFilter data early to reduce the size of the DataFrames before joining.\nPartitioning: Ensure the data is properly partitioned on the join key.\nAvoid Skewness by salting or balancing the data.\nCache intermediate results if they are reused.\nUse SQL hints like /*+ BROADCAST() */ to control the join strategy.\n'

In [6]:
#53. Explain the use of broadcast in PySpark.
'''
Broadcast is used to replicate a small DataFrame or RDD to all executor nodes. 
It minimizes data shuffling by making the smaller dataset available locally for joins or other operations.
'''
broadcast_df = broadcast(customers_df)
joined_df = df1.join(broadcast_df, on="customer_id", how="inner")
joined_df.show()

+-----------+--------+----------+-----------------+----------+--------+-----+-------------+--------+
|customer_id|order_id|order_date|order_purchase_ts|product_id|quantity|price|customer_name|location|
+-----------+--------+----------+-----------------+----------+--------+-----+-------------+--------+
+-----------+--------+----------+-----------------+----------+--------+-----+-------------+--------+



In [8]:
#54. Write a PySpark query to perform a cross join.
cross_join_df = df1.crossJoin(customers_df)
cross_join_df.show(5)

+--------+-----------+----------+-----------------+----------+--------+-----+-----------+---------------+----------+
|order_id|customer_id|order_date|order_purchase_ts|product_id|quantity|price|customer_id|  customer_name|  location|
+--------+-----------+----------+-----------------+----------+--------+-----+-----------+---------------+----------+
|    1001|        501|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|        101|       John Doe|  New York|
|    1001|        501|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|        102|     Jane Smith|California|
|    1001|        501|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|        103|Michael Johnson|     Texas|
|    1001|        501|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|        104|    Emily Davis|   Florida|
|    1001|        501|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|        105|   David Wilson|      Ohio|
+--------+-----------+----------+-----------------+----------+--

In [9]:
#55. How do you perform self-joins in PySpark?
#Self-joins are used to join a DataFrame with itself, usually based on a condition

self_join_df = df1.alias("a").join(
    df1.alias("b"), 
    col("a.customer_id") == col("b.customer_id")
)
self_join_df.show()

+--------+-----------+----------+-----------------+----------+--------+-----+--------+-----------+----------+-----------------+----------+--------+-----+
|order_id|customer_id|order_date|order_purchase_ts|product_id|quantity|price|order_id|customer_id|order_date|order_purchase_ts|product_id|quantity|price|
+--------+-----------+----------+-----------------+----------+--------+-----+--------+-----------+----------+-----------------+----------+--------+-----+
|    1001|        501|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|    1001|        501|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|
|    1002|        502|02-10-2017| 02-10-2017 10:56|      P002|       1| 85.5|    1002|        502|02-10-2017| 02-10-2017 10:56|      P002|       1| 85.5|
|    1003|        503|03-10-2017| 03-10-2017 11:45|      P001|       3|150.0|    1003|        503|03-10-2017| 03-10-2017 11:45|      P001|       3|150.0|
|    1004|        504|04-10-2017| 04-10-2017 14:15|      P003|       1|200.0

In [10]:
#56. How do you perform anti-joins in PySpark?

#An anti-join returns rows from the left DataFrame that do not have matching rows in the right DataFrame.
anti_join_df = df1.join(customers_df, on="customer_id", how="left_anti")
anti_join_df.show()

+-----------+--------+----------+-----------------+----------+--------+-----+
|customer_id|order_id|order_date|order_purchase_ts|product_id|quantity|price|
+-----------+--------+----------+-----------------+----------+--------+-----+
|        501|    1001|01-10-2017| 01-10-2017 09:23|      P001|       2|150.0|
|        502|    1002|02-10-2017| 02-10-2017 10:56|      P002|       1| 85.5|
|        503|    1003|03-10-2017| 03-10-2017 11:45|      P001|       3|150.0|
|        504|    1004|04-10-2017| 04-10-2017 14:15|      P003|       1|200.0|
|        505|    1005|05-10-2017| 05-10-2017 16:40|      P002|       5| 85.5|
|        506|    1006|08-10-2017| 08-10-2017 12:30|      P004|       2|110.0|
|        507|    1007|09-10-2017| 09-10-2017 08:20|      P001|       1|150.0|
|        508|    1008|10-10-2017| 10-10-2017 18:05|      P005|       3| 60.0|
|        509|    1009|11-10-2017| 11-10-2017 09:40|      P003|       2|200.0|
|        510|    1010|12-10-2017| 12-10-2017 11:30|      P002|  

In [11]:
#57. Explain the significance of shuffle operations in PySpark.

'''Shuffle operations involve redistributing data across partitions, which can be expensive due to data movement between nodes. 
Joins, aggregations, and repartitioning can trigger shuffles.

Optimization Tips:

Minimize shuffles by reducing the number of stages.
Use broadcast joins where applicable.
Pre-partition data based on the join key.'''


'Shuffle operations involve redistributing data across partitions, which can be expensive due to data movement between nodes. \nJoins, aggregations, and repartitioning can trigger shuffles.\n\nOptimization Tips:\n\nMinimize shuffles by reducing the number of stages.\nUse broadcast joins where applicable.\nPre-partition data based on the join key.'

In [13]:
#58. How do you perform skew handling in PySpark?

'''
Skew occurs when one partition contains significantly more data than others. Handling skew ensures better parallelism.

Techniques:

Salting: Add a random key to the data to spread it across partitions.
Split large keys: Divide the data for skewed keys.
Broadcast smaller DataFrame: Avoid shuffle by broadcasting.
'''
salted_df = df1.withColumn("salt", lit(1))
joined_df = salted_df.join(customers_df, on=["customer_id", "salt"], how="inner")
joined_df.show()

'\nSkew occurs when one partition contains significantly more data than others. Handling skew ensures better parallelism.\n\nTechniques:\n\nSalting: Add a random key to the data to spread it across partitions.\nSplit large keys: Divide the data for skewed keys.\nBroadcast smaller DataFrame: Avoid shuffle by broadcasting.\n'

In [14]:
#59. Explain the difference between narrow and wide transformations.

'''
Narrow Transformation:

Operations like map and filter where each input partition contributes to only one output partition.
No data shuffle is required.
Wide Transformation:

Operations like groupBy and join that require shuffling data across partitions.
Higher cost due to shuffle.
'''

'\nNarrow Transformation:\n\nOperations like map and filter where each input partition contributes to only one output partition.\nNo data shuffle is required.\nWide Transformation:\n\nOperations like groupBy and join that require shuffling data across partitions.\nHigher cost due to shuffle.\n'

In [15]:
#60. How do you perform an outer join in PySpark?

outer_join_df = df1.join(customers_df, on="customer_id", how="outer")
outer_join_df.show()

+-----------+--------+----------+-----------------+----------+--------+-----+---------------+----------+
|customer_id|order_id|order_date|order_purchase_ts|product_id|quantity|price|  customer_name|  location|
+-----------+--------+----------+-----------------+----------+--------+-----+---------------+----------+
|        101|    NULL|      NULL|             NULL|      NULL|    NULL| NULL|       John Doe|  New York|
|        102|    NULL|      NULL|             NULL|      NULL|    NULL| NULL|     Jane Smith|California|
|        103|    NULL|      NULL|             NULL|      NULL|    NULL| NULL|Michael Johnson|     Texas|
|        104|    NULL|      NULL|             NULL|      NULL|    NULL| NULL|    Emily Davis|   Florida|
|        105|    NULL|      NULL|             NULL|      NULL|    NULL| NULL|   David Wilson|      Ohio|
|        106|    NULL|      NULL|             NULL|      NULL|    NULL| NULL|   Sarah Miller|   Arizona|
|        107|    NULL|      NULL|             NULL|    

In [17]:
#61. How do you read JSON data into a PySpark DataFrame?

json_df = spark.read.json("/user/duggiedogy/data/sample.json", multiLine=True)
json_df.show()

+--------------------+---+-----+--------------------+
|             address| id| name|              orders|
+--------------------+---+-----+--------------------+
|{Springfield, 123...|  1|Alice|[{250, 101}, {300...|
|{Riverside, 456 O...|  2|  Bob|        [{150, 201}]|
+--------------------+---+-----+--------------------+



In [18]:
#62. Write a PySpark query to flatten nested JSON data.
#Flattening involves extracting nested fields into top-level columns using select or explode.
# Assume nested JSON structure
flattened_df = json_df.select(
    col("id"),
    col("name"),
    col("address.street").alias("street"),
    col("address.city").alias("city"),
    explode(col("orders")).alias("order")
)
flattened_df.show()

+---+-----+------------+-----------+----------+
| id| name|      street|       city|     order|
+---+-----+------------+-----------+----------+
|  1|Alice|123 Maple St|Springfield|{250, 101}|
|  1|Alice|123 Maple St|Springfield|{300, 102}|
|  2|  Bob| 456 Oak Ave|  Riverside|{150, 201}|
+---+-----+------------+-----------+----------+



In [19]:
# Create a sample DataFrame
data = [
    (1, "Alice", "123 Maple St", "Springfield", 250),
    (2, "Bob", "456 Oak Ave", "Riverside", 150)
]
columns = ["id", "name", "street", "city", "amount"]
df = spark.createDataFrame(data, columns)

# Write to Parquet
df.write.parquet("/user/duggiedogy/data/sample.parquet", mode="overwrite")

In [20]:
#63. How do you handle schema evolution in PySpark?

'''Schema evolution involves handling changes in the structure of incoming data. 
PySpark can handle schema evolution by enabling options like mergeSchema.'''

schema_evolved_df = spark.read.option("mergeSchema", "true").parquet("/user/duggiedogy/data/sample.parquet")
schema_evolved_df.show()

+---+-----+------------+-----------+------+
| id| name|      street|       city|amount|
+---+-----+------------+-----------+------+
|  1|Alice|123 Maple St|Springfield|   250|
|  2|  Bob| 456 Oak Ave|  Riverside|   150|
+---+-----+------------+-----------+------+



In [2]:
#66. How do you process large datasets efficiently using PySpark?

'''
Techniques:

Partitioning: Optimize data partitioning to balance workload.
Broadcast Joins: Use broadcast for small DataFrames.
Cache and Persist: Cache intermediate results if reused.
Avoid Wide Transformations: Minimize shuffle operations.
Use Columnar Formats: Use Parquet/ORC for efficient I/O.
'''

'\nTechniques:\n\nPartitioning: Optimize data partitioning to balance workload.\nBroadcast Joins: Use broadcast for small DataFrames.\nCache and Persist: Cache intermediate results if reused.\nAvoid Wide Transformations: Minimize shuffle operations.\nUse Columnar Formats: Use Parquet/ORC for efficient I/O.\n'

In [3]:
#67. Explain how to use PySpark UDFs (User Defined Functions):
'''PySpark UDFs are custom functions written in Python that can be applied to PySpark DataFrame columns. 
These functions allow you to execute non-native operations, including complex logic, on your DataFrame. 
UDFs must be registered with the pyspark.sql.functions.udf API and can work with scalar values or structured types.

Steps to Use UDFs:

Define a Python function with the desired logic.
Register the function as a UDF using udf() and specify the return type.
Apply the UDF to a DataFrame column using the withColumn or select methods.'''

'PySpark UDFs are custom functions written in Python that can be applied to PySpark DataFrame columns. \nThese functions allow you to execute non-native operations, including complex logic, on your DataFrame. \nUDFs must be registered with the pyspark.sql.functions.udf API and can work with scalar values or structured types.\n\nSteps to Use UDFs:\n\nDefine a Python function with the desired logic.\nRegister the function as a UDF using udf() and specify the return type.\nApply the UDF to a DataFrame column using the withColumn or select methods.'

In [5]:
#68. Write a PySpark UDF to Clean and Normalize Text Data:

from pyspark.sql.functions import udf
from pyspark.sql.types import StringType
import re

# Define a Python function
def clean_text(text):
    if text:
        text = text.lower()  # Convert to lowercase
        text = re.sub(r"[^a-zA-Z0-9\s]", "", text)  # Remove special characters
        return text.strip()  # Remove leading/trailing spaces
    return text

# Register the UDF
clean_text_udf = udf(clean_text, StringType())

# Example DataFrame
data = [("Alice@",), ("  Bob!!",), ("   CHARLIE###",)]
columns = ["name"]
df = spark.createDataFrame(data, columns)

# Apply UDF
cleaned_df = df.withColumn("cleaned_name", clean_text_udf("name"))
cleaned_df.show()

+-------------+------------+
|         name|cleaned_name|
+-------------+------------+
|       Alice@|       alice|
|        Bob!!|         bob|
|   CHARLIE###|     charlie|
+-------------+------------+



In [6]:
#69. How Do You Implement ETL Pipelines in PySpark?

'''
An ETL (Extract, Transform, Load) pipeline in PySpark involves the following steps:

Extract: Read data from sources like CSV, JSON, Parquet, or databases using spark.read.
Transform: Apply transformations like filtering, aggregation, and joining using PySpark DataFrame operations.
Load: Save the transformed data to a destination like a data warehouse, storage system, or database using write methods.
'''

'\nAn ETL (Extract, Transform, Load) pipeline in PySpark involves the following steps:\n\nExtract: Read data from sources like CSV, JSON, Parquet, or databases using spark.read.\nTransform: Apply transformations like filtering, aggregation, and joining using PySpark DataFrame operations.\nLoad: Save the transformed data to a destination like a data warehouse, storage system, or database using write methods.\n'

In [ ]:
#70. How Do You Read Data from Multiple Files in PySpark?

'''
You can read data from multiple files by specifying:

A list of file paths.
A directory containing multiple files.
Using wildcard patterns in file paths.
'''
#read specific files
df = spark.read.csv(["file1.csv", "file2.csv"], header=True, inferSchema=True)
df.show()

#read all files in directory
df = spark.read.csv("path/to/directory/", header=True, inferSchema=True)
df.show()

#read Files Using Wildcard Patterns
df = spark.read.csv("path/to/files/prefix-*.csv", header=True, inferSchema=True)
df.show()

In [ ]:
#71. How Do You Handle Schema Drift in PySpark?

'''
Schema drift occurs when the schema of incoming data changes over time. PySpark can handle schema drift using:

Schema Inference: Automatically detects schema using inferSchema=True.
Explicit Schemas: Define a schema to enforce consistency.
Schema Merge: Enable schema merging for formats like Parquet or Avro when reading data with different schemas.
'''
#Schema Merge in Parquet
df = spark.read.option("mergeSchema", "true").parquet("path/to/parquet/files")
df.printSchema()

In [3]:
#Handling with Explicit Schemas
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

schema = StructType([
    StructField("id", IntegerType(), True),
    StructField("name", StringType(), True),
    StructField("age", IntegerType(), True)
])

df = spark.read.schema(schema).json("path/to/json/file")
df.show()

AnalysisException: [PATH_NOT_FOUND] Path does not exist: hdfs://dataproc-spark-m/user/root/path/to/json/file.

In [4]:
#72. Write a PySpark Query to Pivot and Unpivot Data
#Pivoting: Convert rows into columns.

data = [("Alice", "Math", 85), ("Alice", "English", 90), 
        ("Bob", "Math", 78), ("Bob", "English", 88)]
columns = ["name", "subject", "score"]

df = spark.createDataFrame(data, columns)

pivot_df = df.groupBy("name").pivot("subject").sum("score")
pivot_df.show()

#Unpivoting: Convert columns into rows.
from pyspark.sql.functions import expr

unpivot_df = pivot_df.selectExpr("name", "stack(2, 'Math', Math, 'English', English) as (subject, score)")
unpivot_df.show()

+-----+-------+----+
| name|English|Math|
+-----+-------+----+
|Alice|     90|  85|
|  Bob|     88|  78|
+-----+-------+----+

+-----+-------+-----+
| name|subject|score|
+-----+-------+-----+
|Alice|   Math|   85|
|Alice|English|   90|
|  Bob|   Math|   78|
|  Bob|English|   88|
+-----+-------+-----+



In [5]:
#73. How Do You Explode an Array or Map Column in PySpark?

#The explode function flattens array or map columns into separate rows.

#array example
data = [("Alice", [1, 2, 3]), ("Bob", [4, 5])]
columns = ["name", "scores"]

df = spark.createDataFrame(data, columns)
exploded_df = df.select("name", expr("explode(scores) as score"))
exploded_df.show()

+-----+-----+
| name|score|
+-----+-----+
|Alice|    1|
|Alice|    2|
|Alice|    3|
|  Bob|    4|
|  Bob|    5|
+-----+-----+



In [6]:
#map example
data = [("Alice", {"Math": 85, "English": 90}), ("Bob", {"Math": 78, "English": 88})]
columns = ["name", "subjects"]

df = spark.createDataFrame(data, columns)
exploded_df = df.select("name", expr("explode(subjects) as (subject, score)"))
exploded_df.show()

+-----+-------+-----+
| name|subject|score|
+-----+-------+-----+
|Alice|   Math|   85|
|Alice|English|   90|
|  Bob|   Math|   78|
|  Bob|English|   88|
+-----+-------+-----+



In [7]:
#74. How Do You Convert Rows into Columns (Pivoting)?
#Pivoting rows into columns is achieved using the pivot() function. 

In [8]:
#75. Write a PySpark Query to Transform Semi-Structured Data (XML, JSON)
from pyspark.sql.functions import col, json_tuple

data = [("{'id':1, 'name':'Alice'}",), ("{'id':2, 'name':'Bob'}",)]
columns = ["json_data"]

df = spark.createDataFrame(data, columns)
transformed_df = df.select(json_tuple(col("json_data"), "id", "name").alias("id", "name"))
transformed_df.show()

+---+-----+
| id| name|
+---+-----+
|  1|Alice|
|  2|  Bob|
+---+-----+



In [ ]:
#Processing XML For XML, PySpark requires an external library like com.databricks.spark.xml.
df = spark.read.format("com.databricks.spark.xml").option("rowTag", "book").load("path/to/books.xml")
df.show()

## High Complexity 25 Questions

In [9]:
#76. How do you optimize PySpark jobs to handle large datasets?

'''Partitioning: Ensure your data is properly partitioned to allow for parallel processing.
Broadcast joins: Use broadcast() for small DataFrames to avoid shuffles during joins.

Caching: Cache intermediate results using .cache() or .persist() to avoid recomputation.

Filter early: Apply filters and transformations as early as possible in the job to reduce the size of the dataset.

Avoid wide transformations: Minimize shuffles caused by operations like groupBy, join, and reduceByKey.

Use DataFrame API: The DataFrame API is optimized for performance compared to RDDs.

Dynamic allocation: Use Spark's dynamic allocation to scale resources based on workload.'''

"Partitioning: Ensure your data is properly partitioned to allow for parallel processing.\nBroadcast joins: Use broadcast() for small DataFrames to avoid shuffles during joins.\n\nCaching: Cache intermediate results using .cache() or .persist() to avoid recomputation.\n\nFilter early: Apply filters and transformations as early as possible in the job to reduce the size of the dataset.\n\nAvoid wide transformations: Minimize shuffles caused by operations like groupBy, join, and reduceByKey.\n\nUse DataFrame API: The DataFrame API is optimized for performance compared to RDDs.\n\nDynamic allocation: Use Spark's dynamic allocation to scale resources based on workload."

In [10]:
#77. Explain how Spark's Catalyst optimizer works.

'''
The Catalyst Optimizer is a query optimization engine used in Spark SQL. 
It optimizes logical and physical plans to improve query performance. Key components:

Analysis: Validates the query plan and resolves references to columns and tables.

Logical optimization: Applies rules like predicate pushdown and column pruning to minimize data processed.

Physical planning: Converts the logical plan into multiple physical plans and selects the most efficient one based on cost.

Code generation: Uses whole-stage code generation to produce optimized JVM bytecode for execution.
'''

'\nThe Catalyst Optimizer is a query optimization engine used in Spark SQL. \nIt optimizes logical and physical plans to improve query performance. Key components:\n\nAnalysis: Validates the query plan and resolves references to columns and tables.\n\nLogical optimization: Applies rules like predicate pushdown and column pruning to minimize data processed.\n\nPhysical planning: Converts the logical plan into multiple physical plans and selects the most efficient one based on cost.\n\nCode generation: Uses whole-stage code generation to produce optimized JVM bytecode for execution.\n'

In [11]:
#78. How do you handle memory optimization in PySpark?

'''
Broadcast smaller datasets: Reduce memory usage during joins by broadcasting smaller DataFrames.

Compression: Enable compression for shuffle and storage (e.g., spark.sql.parquet.compression.codec set to snappy).

Avoid large collect(): Don’t use .collect() on large datasets to prevent driver memory issues.

Use appropriate partitioning: Avoid excessive small partitions or very large partitions.

Garbage collection tuning: Configure JVM GC parameters for efficient memory management.

Persist with disk: Use .persist(StorageLevel.DISK_ONLY) for very large intermediate results.

Monitor Spark UI: Use the storage tab to track memory usage and optimize persist levels.
'''

'\nBroadcast smaller datasets: Reduce memory usage during joins by broadcasting smaller DataFrames.\n\nCompression: Enable compression for shuffle and storage (e.g., spark.sql.parquet.compression.codec set to snappy).\n\nAvoid large collect(): Don’t use .collect() on large datasets to prevent driver memory issues.\n\nUse appropriate partitioning: Avoid excessive small partitions or very large partitions.\n\nGarbage collection tuning: Configure JVM GC parameters for efficient memory management.\n\nPersist with disk: Use .persist(StorageLevel.DISK_ONLY) for very large intermediate results.\n\nMonitor Spark UI: Use the storage tab to track memory usage and optimize persist levels.\n'

In [18]:
#79. Write a PySpark query to handle out-of-memory errors.

'''
Out-of-memory errors often occur due to operations requiring large memory allocations. Strategies:

Use repartition() or coalesce() to manage partition sizes.
'''
from pyspark.storagelevel import StorageLevel

# Repartition data to avoid large partitions
large_df = large_df.repartition(200)

# Persist to disk to handle memory pressure
large_df.persist(StorageLevel.DISK_ONLY)

# Perform operations
result = large_df.groupBy("category").sum("value")
result.show()

In [12]:
#80. How do you profile PySpark jobs to identify bottlenecks?
'''
Spark UI: Monitor jobs and stages through the Spark UI. Look for:
Skewed tasks.
Long-running stages.
Shuffle and GC overhead.

Event Logs: Enable Spark event logging (spark.eventLog.enabled=true) and analyze logs using tools like Dr. Elephant.

Metrics system: Configure Spark metrics to capture detailed runtime stats.

Job instrumentation: Add timers to stages using .time() or time.time() in Python.

Data sampling: Run your job on a small subset of data to identify problematic operations early.

Use cache profiler: Check the effect of caching intermediate results and storage behavior.

Optimize transformations: Inspect the DAG (Directed Acyclic Graph) in the Spark UI to spot expensive transformations.
'''

'\nSpark UI: Monitor jobs and stages through the Spark UI. Look for:\nSkewed tasks.\nLong-running stages.\nShuffle and GC overhead.\n\nEvent Logs: Enable Spark event logging (spark.eventLog.enabled=true) and analyze logs using tools like Dr. Elephant.\n\nMetrics system: Configure Spark metrics to capture detailed runtime stats.\n\nJob instrumentation: Add timers to stages using .time() or time.time() in Python.\n\nData sampling: Run your job on a small subset of data to identify problematic operations early.\n\nUse cache profiler: Check the effect of caching intermediate results and storage behavior.\n\nOptimize transformations: Inspect the DAG (Directed Acyclic Graph) in the Spark UI to spot expensive transformations.\n'

In [ ]:
#81. Write a PySpark query to join 10 large tables efficiently.

'''When joining large tables, optimizations such as broadcast joins, partitioning, and avoiding shuffles are crucial.

Steps:

Identify small tables and use broadcast join.
Repartition large tables to ensure balanced workload.
Avoid joining all tables at once; join in pairs.'''
from pyspark.sql.functions import *

# Broadcast small tables
table1 = spark.read.parquet("table1.parquet")
table2 = broadcast(spark.read.parquet("table2.parquet"))

# Join tables incrementally
joined_table = table1.join(table2, "common_key")

for i in range(3, 11):  # Iteratively join tables 3 to 10
    next_table = spark.read.parquet(f"table{i}.parquet")
    joined_table = joined_table.join(next_table, "common_key")

joined_table.show()


In [ ]:
#82. How do you handle skewed data in PySpark?

'''Skewed data leads to unbalanced partitions and processing delays. Strategies:
Salting: Add a random prefix to keys to distribute data across partitions.

Broadcast small datasets: Broadcast the smaller dataset to avoid shuffles during joins.

Repartition skewed keys: Repartition using .repartition() or .coalesce() with non-skewed keys.

Increase shuffle partitions: Adjust the number of partitions using spark.sql.shuffle.partitions.
'''
from pyspark.sql.functions import lit, concat

salted_df = large_df.withColumn("salted_key", concat(lit("salt_"), col("key")))

In [ ]:
#83. Implement a PySpark job for incremental data processing.

#Incremental processing handles only new or updated data rather than reprocessing all data.

from pyspark.sql.functions import col, max

# Load current data and previous snapshot
current_data = spark.read.parquet("current_data.parquet")
previous_data = spark.read.parquet("previous_snapshot.parquet")

# Find maximum timestamp in previous snapshot
max_timestamp = previous_data.agg(max("update_time")).collect()[0][0]

# Filter new data
new_data = current_data.filter(col("update_time") > max_timestamp)

# Combine new data with previous data
updated_data = previous_data.union(new_data)

# Save updated snapshot
updated_data.write.parquet("updated_snapshot.parquet", mode="overwrite")

In [ ]:
#84. How do you handle late-arriving data in PySpark ETL pipelines?

'''Late-arriving data refers to data that arrives after the processing window has closed.
Watermarking: Use watermarking to handle late data within a defined time limit (for streaming jobs).
'''
#watermarking
streaming_df = spark.readStream.format("kafka").option("subscribe", "topic").load()
streaming_df = streaming_df.withWatermark("event_time", "1 hour")

'''
Buffering: Store late-arriving data in a temporary location for reprocessing later.
Reprocessing pipelines: Rerun pipelines with late data and merge with existing results.
Use event time: Always process data based on the event time instead of the ingestion time.

'''

In [ ]:
#85. Write a PySpark code to perform deduplication on billions of records.

#Efficient deduplication involves identifying duplicates based on a unique key and keeping only one record for each key.

from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Define window partitioned by unique key
window_spec = Window.partitionBy("unique_key").orderBy("timestamp")

# Assign row numbers to each record within the partition
deduplicated_df = large_df.withColumn("row_num", row_number().over(window_spec)).filter(col("row_num") == 1)

# Drop the row_num column and save the results
deduplicated_df = deduplicated_df.drop("row_num")
deduplicated_df.write.parquet("deduplicated_data.parquet", mode="overwrite")

In [ ]:
#86. Implement a PySpark job to process streaming data.

from pyspark.sql import SparkSession
from pyspark.sql.functions import col

# Initialize Spark Session with streaming support
spark = SparkSession.builder \
    .appName("StreamingExample") \
    .getOrCreate()

# Read streaming data
streaming_df = spark.readStream \
    .format("csv") \
    .option("header", "true") \
    .schema("sensor_id INT, temperature FLOAT, timestamp STRING") \
    .load("path/to/streaming/files")

# Apply transformations
processed_df = streaming_df.filter(col("temperature") > 30)

# Write output to console
query = processed_df.writeStream \
    .outputMode("append") \
    .format("console") \
    .start()

query.awaitTermination()

In [ ]:
#87. How do you integrate PySpark with Kafka?

'''
Integration Steps:

Add Kafka dependencies to your Spark application.
Use the readStream API to connect to a Kafka topic.
'''

from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StringType, IntegerType

# Define schema for Kafka messages
schema = StructType([
    col("sensor_id").cast(IntegerType()),
    col("temperature").cast(StringType()),
    col("timestamp").cast(StringType())
])

# Initialize Spark session
spark = SparkSession.builder.appName("KafkaIntegration").getOrCreate()

# Read from Kafka
kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "sensor-data") \
    .load()

# Parse Kafka messages
parsed_df = kafka_df.selectExpr("CAST(value AS STRING)") \
    .select(from_json(col("value"), schema).alias("data")) \
    .select("data.*")

parsed_df.writeStream \
    .format("console") \
    .outputMode("append") \
    .start() \
    .awaitTermination()


In [ ]:
#88. Write a PySpark job to clean and aggregate IoT sensor data.

from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col

# Initialize Spark session
spark = SparkSession.builder.appName("IoTDataProcessing").getOrCreate()

# Read IoT sensor data
iot_data = spark.read.csv("path/to/sensor_data.csv", header=True, inferSchema=True)

# Clean data: Remove nulls and filter valid temperatures
clean_data = iot_data.filter((col("temperature").isNotNull()) & (col("temperature") > 0))

# Aggregate data: Calculate average temperature per sensor
aggregated_data = clean_data.groupBy("sensor_id").agg(avg("temperature").alias("avg_temperature"))

aggregated_data.show()

In [ ]:
#89. How do you implement slowly changing dimensions (SCD) in PySpark?

from pyspark.sql import SparkSession
from pyspark.sql.functions import col, current_date, lit, when

# Initialize Spark session
spark = SparkSession.builder.appName("SCDType2").getOrCreate()

# Load current and new data
current_data = spark.read.parquet("path/to/current_data.parquet")
new_data = spark.read.csv("path/to/new_data.csv", header=True, inferSchema=True)

# Join current and new data to identify changes
scd_df = new_data.join(current_data, on="id", how="outer") \
    .withColumn("is_new", col("current.id").isNull()) \
    .withColumn("is_changed", col("new_data.value") != col("current_data.value"))

# Mark old records as expired
updated_current = scd_df.filter(col("is_changed") | col("is_new")) \
    .withColumn("end_date", current_date())

# Insert new records with current status
new_records = scd_df.filter(col("is_new")) \
    .withColumn("start_date", current_date()) \
    .withColumn("end_date", lit(None))

# Merge records back
final_df = updated_current.union(new_records)
final_df.write.parquet("path/to/final_scd.parquet", mode="overwrite")

In [ ]:
#90. Write a PySpark job to process time-series data.

from pyspark.sql import SparkSession
from pyspark.sql.functions import window, avg, col

# Initialize Spark session
spark = SparkSession.builder.appName("TimeSeriesProcessing").getOrCreate()

# Read time-series data
time_series_data = spark.read.csv("path/to/timeseries_data.csv", header=True, inferSchema=True)

# Aggregate data: Calculate hourly average
windowed_data = time_series_data \
    .groupBy(window(col("timestamp"), "1 hour")) \
    .agg(avg("value").alias("avg_value"))

windowed_data.show()